# Wide-binary ṽ(g_N) across the low-acceleration boundary (ORB-10753)

Predeclared baseline (locked in the task plan *before* binning):
- Quality: `ruwe < 1.4`, `parallax_over_error > 10`, `G < 18`, `ϖ > 5` mas (d < 200 pc)
- Pair geometry: θ ∈ [1.5″, 1°], s ∈ [0.5, 50] kau; parallax + PM consistency (El-Badry style)
- Eccentricity prior for Newtonian mocks: **thermal** f(e)=2e
- Triple policy: RUWE veto + |ΔRV|<20 km/s; residual f_triple ≈ 0.10
- Chance alignment: shifted-field RA +0.5°

This notebook loads Store products written by `scripts/run_wide_binary_study.py`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from astrolabe.store import Store

store = Store('data')
print('datasets:', store.datasets())
pairs = store.read('widebin_pairs_baseline')
binned = store.read('widebin_vtilde_gn')
sens = store.read('widebin_sensitivity')
meta_pairs = store.read_meta('widebin_pairs_baseline')
print(f'pairs={len(pairs)} bins={len(binned)}')
print('cuts:', json.dumps(meta_pairs.query.get('baseline_cuts', {}), indent=2)[:500])


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
g = np.asarray(binned['g_N_mid_ms2'], dtype=float)
vt = np.asarray(binned['vtilde_med'], dtype=float)
err = np.asarray(binned['vtilde_err'], dtype=float)
m = np.asarray(binned['vtilde_mock_med'], dtype=float)
lo = np.asarray(binned['vtilde_mock_lo'], dtype=float)
hi = np.asarray(binned['vtilde_mock_hi'], dtype=float)
ax.errorbar(g, vt, yerr=err, fmt='o', label='data median ṽ', color='C0')
ax.fill_between(g, lo, hi, color='C1', alpha=0.25, label='Newtonian mock p16–p84')
ax.plot(g, m, 's--', color='C1', label='Newtonian mock median')
ax.axvline(1.2e-10, color='k', ls=':', label=r'$a_0 \approx 1.2\times10^{-10}$')
ax.set_xscale('log')
ax.set_xlabel(r'$g_N$ [m s$^{-2}$]')
ax.set_ylabel(r'$\tilde{v}$')
ax.set_title('Scaled wide-binary velocity vs internal Newtonian acceleration')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
Path('data/scratch').mkdir(parents=True, exist_ok=True)
fig.savefig('data/scratch/widebin_vtilde_gn.png', dpi=120)
print('wrote data/scratch/widebin_vtilde_gn.png')
binned


In [ ]:
print('Sensitivity table (ecc prior × RUWE cut × regime):')
sens

## Provenance notes for principia

- Gaia DR3 DOI: `10.5270/esa-1ugzkg7`
- ADQL + cuts live in `catalog/widebin_gaia_dr3_d200` and `catalog/widebin_pairs_baseline` sidecars
- Selection style: El-Badry, Rix & Heintz 2021; **not** a re-download of their catalog
- Masses: Pecaut & Mamajek 2013-inspired M_G → M (Gaia G)
- Comparison is forward-modeled Newtonian MC with the same projected-s window and ecc prior — not circular-orbit analytic
- Contested modeling (Chae vs Banik) is tabulated in `derived/widebin_sensitivity`; this pipeline does not adjudicate the literature